In [ ]:
# ============================================
# IMPORTS
# ============================================

import os
import json
import numpy as np
import tensorflow as tf

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import (
    Dense,
    Dropout,
    GlobalAveragePooling2D
)

from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam

from sklearn.metrics import classification_report, f1_score


# ============================================
# LOAD DATA
# ============================================

X_img_train = np.load("data/X_img_train.npy")
X_img_val   = np.load("data/X_img_val.npy")
X_img_test  = np.load("data/X_img_test.npy")

y_train = np.load("data/y_train.npy")
y_val   = np.load("data/y_val.npy")
y_test  = np.load("data/y_test.npy")

print("Train Images:", X_img_train.shape)
print("Validation Images:", X_img_val.shape)
print("Test Images:", X_img_test.shape)


# ============================================
# LOAD LABELS
# ============================================

with open("data/label_classes.json") as f:
    label_classes = json.load(f)

NUM_CLASSES = len(label_classes)

print("\nClasses:")
print(label_classes)


# ============================================
# ONE HOT ENCODING
# ============================================

y_train_cat = to_categorical(y_train, NUM_CLASSES)
y_val_cat   = to_categorical(y_val, NUM_CLASSES)
y_test_cat  = to_categorical(y_test, NUM_CLASSES)


# ============================================
# BUILD MODEL
# ============================================

base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# IMPORTANT
base_model.trainable = True

model = Sequential([

    base_model,

    GlobalAveragePooling2D(),

    Dense(256, activation="relu"),

    Dropout(0.3),

    Dense(NUM_CLASSES, activation="softmax")

])


# ============================================
# COMPILE
# ============================================

model.compile(

    optimizer=Adam(learning_rate=0.0001),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

model.summary()


# ============================================
# EARLY STOPPING
# ============================================

early_stop = EarlyStopping(

    patience=3,

    restore_best_weights=True

)


# ============================================
# TRAIN
# ============================================

history = model.fit(

    X_img_train,
    y_train_cat,

    validation_data=(

        X_img_val,
        y_val_cat

    ),

    epochs=20,

    batch_size=16,

    callbacks=[early_stop]

)


# ============================================
# EVALUATE
# ============================================

y_pred_probs = model.predict(X_img_test)

y_pred = np.argmax(y_pred_probs, axis=1)

f1 = f1_score(

    y_test,

    y_pred,

    average="weighted"

)

print("\nF1 Score:", f1)


print("\nClassification Report:\n")

print(

    classification_report(

        y_test,

        y_pred,

        target_names=list(label_classes.values())

    )

)


# ============================================
# SAVE MODEL
# ============================================

model.save("image_classifier.h5")

print("\nModel Saved Successfully ")




Train Images: (637, 224, 224, 3)
Validation Images: (80, 224, 224, 3)
Test Images: (80, 224, 224, 3)

Classes:
{'0': 'amoxicillin', '1': 'aspirin', '2': 'ibuprofen', '3': 'insulin', '4': 'omeprazole', '5': 'paracetamol'}


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │         1,542 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,587,462 (9.87 MB)

 Trainable params: 2,553,350 (9.74 MB)

 Non-trainable params: 34,112 (133.25 KB)

Epoch 1/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 173s 2s/step - accuracy: 0.4929 - loss: 1.3473 - val_accuracy: 0.4125 - val_loss: 1.3593
Epoch 2/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 84s 2s/step - accuracy: 0.8462 - loss: 0.5087 - val_accuracy: 0.5750 - val_loss: 1.1807
Epoch 3/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 84s 2s/step - accuracy: 0.9419 - loss: 0.2390 - val_accuracy: 0.6375 - val_loss: 1.0467
Epoch 4/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 83s 2s/step - accuracy: 0.9874 - loss: 0.1026 - val_accuracy: 0.6375 - val_loss: 0.9465
Epoch 5/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 79s 2s/step - accuracy: 0.9953 - loss: 0.0525 - val_accuracy: 0.7000 - val_loss: 0.8847
Epoch 6/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 77s 2s/step - accuracy: 0.9953 - loss: 0.0441 - val_accuracy: 0.7125 - val_loss: 0.7776
Epoch 7/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 77s 2s/step - accuracy: 0.9984 - loss: 0.0238 - val_accuracy: 0.7250 - val_loss: 0.7325
Epoch 8/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 74s 2s/step - accuracy: 0.9984 - loss: 0.0195 - val_accuracy: 0.7375 - val_loss


F1 Score: 0.8724475524475525

Classification Report:

              precision    recall  f1-score   support

 amoxicillin       1.00      0.86      0.92        14
     aspirin       0.83      1.00      0.91        10
   ibuprofen       1.00      0.57      0.73        14
     insulin       1.00      1.00      1.00        12
  omeprazole       0.67      1.00      0.80        12
 paracetamol       0.89      0.89      0.89        18

    accuracy                           0.88        80
   macro avg       0.90      0.89      0.87        80
weighted avg       0.90      0.88      0.87        80


Model Saved Successfully ✅


In [ ]:
# ============================================
# TEST SINGLE IMAGE
# ============================================

from keras.preprocessing import image

img_path = "test_images/test1.png"

img = image.load_img(

    img_path,

    target_size=(224, 224)

)

img_array = image.img_to_array(img) / 255.0

img_array = np.expand_dims(img_array, axis=0)


# ============================================
# PREDICT
# ============================================

pred = model.predict(img_array)

pred_class = np.argmax(pred)

predicted = label_classes[str(pred_class)]

print("\nPrediction:", predicted)


# ============================================
# LOAD SAFETY DATA
# ============================================

with open("data/safety_dataset.json") as f:

    safety_data = json.load(f)


# ============================================
# SAFETY CHECK FUNCTION
# ============================================

def check_safety(drug, condition):

    found = False

    for item in safety_data:

        text = item["input"].lower()

        if drug.lower() in text and condition.lower() in text:

            print("\nSafety Result:")

            print(item["output"])

            found = True

            break

    if not found:

        print("\nNo safety information found")


# ============================================
# USER CONDITION
# ============================================

condition = "headache"

check_safety(predicted, condition)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step

Prediction: aspirin

Safety Result:
safety: safe; recommendation: pain relief; alternative: paracetamol
